In [ ]:
# Improving the Security of Web Services by Automatic Robot Detection
# Authors: Abbas Karimi, Hedieh Sajedi, Abolfazl Khojasteh Abkenar

!pip -q install --upgrade xgboost scikit-learn scipy pandas matplotlib seaborn

import os
import time
import json
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_auc_score, average_precision_score,
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#333333",
    "svg.fonttype": "none",
})
COLORS = {"XGBoost": "#2E5EAA", "RandomForest": "#E8871E", "HistGB": "#3AA655"}

OUT_DIR = "/content/revision_outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def save_table(df, name, index=False):
    path = os.path.join(TAB_DIR, f"{name}.csv")
    df.to_csv(path, index=index)
    print(f"  -> saved {path}")
    return df


def save_fig(fig, name):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, bbox_inches="tight")
    print(f"  -> saved {path}")
    plt.close(fig)


print("\n" + "=" * 80)
print("STEP 1 — LOADING DATA")
print("=" * 80)

DATA_DIR = "/content"  
path_universal = os.path.join(DATA_DIR, "simple_features.csv")     # UNIVERSAL dataset
path_dependent = os.path.join(DATA_DIR, "semantic_features.csv")   # DEPENDENT dataset

if not (os.path.exists(path_universal) and os.path.exists(path_dependent)):
    try:
        from google.colab import files
        print("CSV files not found in", DATA_DIR, "- please upload them now.")
        uploaded = files.upload()
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {path_universal} or {path_dependent}. "
            "Upload the CSVs to Colab or update DATA_DIR."
        )

universal_df = pd.read_csv(path_universal)
dependent_df = pd.read_csv(path_dependent)

ID_COL, TARGET_COL = "ID", "ROBOT"
universal_features = [c for c in universal_df.columns if c not in (ID_COL, TARGET_COL)]
dependent_features = [c for c in dependent_df.columns if c not in (ID_COL, TARGET_COL)]

DATASETS = {
    "Universal": (universal_df, universal_features),
    "Dependent": (dependent_df, dependent_features),
}

print(f"Universal dataset : {universal_df.shape[0]:,} sessions, "
      f"{len(universal_features)} features "
      f"{len(universal_features)}, see revision guide)")
print(f"Dependent dataset : {dependent_df.shape[0]:,} sessions, "
      f"{len(dependent_features)} features")
print(f"Class balance (both datasets share the same {TARGET_COL} labels): "
      f"{universal_df[TARGET_COL].value_counts().to_dict()} "
      f"(0 = human, 1 = robot)")



print("\n" + "=" * 80)
print("STEP 2 — TRAIN/TEST SPLIT")
print("=" * 80)

splits = {}
split_rows = []
for name, (df, feats) in DATASETS.items():
    X, y = df[feats], df[TARGET_COL]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )
    splits[name] = (X_train, X_test, y_train, y_test)
    split_rows.append({
        "Dataset": name,
        "Total sessions": len(df),
        "Train sessions (80%)": len(X_train),
        "Test sessions (20%)": len(X_test),
        "Robot % in train": round(100 * y_train.mean(), 2),
        "Robot % in test": round(100 * y_test.mean(), 2),
    })
    print(f"{name:10s}: total={len(df):,}  train={len(X_train):,}  "
          f"test={len(X_test):,}  (exact 20% = {round(0.2*len(df)):,})")

split_report = pd.DataFrame(split_rows)
save_table(split_report, "TableS1_split_reconciliation")



def build_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),  
        ("scaler", MinMaxScaler()),                            
        ("model", model),
    ])


MODEL_FACTORY = {
    "XGBoost": lambda: XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "HistGB": lambda: HistGradientBoostingClassifier(
        learning_rate=0.1, random_state=RANDOM_STATE,
    ),
}


hp_rows = [
    {"Model": "XGBoost", "Parameter": "n_estimators", "Value": 200},
    {"Model": "XGBoost", "Parameter": "max_depth", "Value": 6},
    {"Model": "XGBoost", "Parameter": "learning_rate", "Value": 0.1},
    {"Model": "RandomForest", "Parameter": "n_estimators", "Value": 100},
    {"Model": "RandomForest", "Parameter": "max_depth", "Value": 10},
    {"Model": "HistGB", "Parameter": "learning_rate", "Value": 0.1},
]
save_table(pd.DataFrame(hp_rows), "Table3_hyperparameters")


print("\n" + "=" * 80)
print("STEP 3 — 5-FOLD CROSS-VALIDATION")
print("=" * 80)

cv_scores = {}  
for dname, (df, feats) in DATASETS.items():
    X, y = df[feats], df[TARGET_COL]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores[dname] = {}
    for mname, factory in MODEL_FACTORY.items():
        pipe = build_pipeline(factory())
        scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
        cv_scores[dname][mname] = scores
        print(f"{dname:10s} | {mname:12s} | folds = {np.round(scores, 4).tolist()} "
              f"| mean = {scores.mean():.4f} | std = {scores.std():.4f}")

table4_rows = []
for dname in DATASETS:
    for mname in MODEL_FACTORY:
        s = cv_scores[dname][mname]
        table4_rows.append({
            "Dataset": dname, "Model": mname,
            "Mean CV Accuracy": round(s.mean(), 4),
            "Std Dev": round(s.std(), 4),
            "Fold 1": round(s[0], 4), "Fold 2": round(s[1], 4),
            "Fold 3": round(s[2], 4), "Fold 4": round(s[3], 4), "Fold 5": round(s[4], 4),
        })
table4 = save_table(pd.DataFrame(table4_rows), "Table4_CV_results")

print("\n" + "-" * 80)
print("STEP 4 — STATISTICAL SIGNIFICANCE TESTING")
print("-" * 80)

sig_rows = []
for dname in DATASETS:
    xgb_scores = cv_scores[dname]["XGBoost"]
    for other in ("RandomForest", "HistGB"):
        other_scores = cv_scores[dname][other]
        t_stat, t_p = stats.ttest_rel(xgb_scores, other_scores)
        try:
            w_stat, w_p = stats.wilcoxon(xgb_scores, other_scores)
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        sig_rows.append({
            "Dataset": dname,
            "Comparison": f"XGBoost vs {other}",
            "Mean Accuracy Diff": round(float(xgb_scores.mean() - other_scores.mean()), 4),
            "Paired t-statistic": round(float(t_stat), 4),
            "t-test p-value": round(float(t_p), 4),
            "Wilcoxon statistic": w_stat if np.isnan(w_stat) else round(float(w_stat), 4),
            "Wilcoxon p-value": w_p if np.isnan(w_p) else round(float(w_p), 4),
            "Significant at alpha=0.05": "Yes" if t_p < 0.05 else "No",
        })
table13 = save_table(pd.DataFrame(sig_rows), "Table13_statistical_significance")
print(table13.to_string(index=False))

print("\n" + "=" * 80)
print("STEP 5 — FINAL TRAIN/TEST EVALUATION (expanded metrics + efficiency)")
print("=" * 80)

results = {}
for dname, (df, feats) in DATASETS.items():
    X_train, X_test, y_train, y_test = splits[dname]
    results[dname] = {}
    for mname, factory in MODEL_FACTORY.items():
        pipe = build_pipeline(factory())

        t0 = time.time()
        pipe.fit(X_train, y_train)
        train_time_s = time.time() - t0

        t0 = time.time()
        y_pred = pipe.predict(X_test)
        infer_time_s = time.time() - t0
        latency_ms = 1000.0 * infer_time_s / len(X_test)

        y_proba = pipe.predict_proba(X_test)[:, 1]
        train_acc = accuracy_score(y_train, pipe.predict(X_train))
        test_acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn)
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)

        results[dname][mname] = dict(
            train_accuracy=train_acc, test_accuracy=test_acc,
            precision_0=prec[0], recall_0=rec[0], f1_0=f1[0], n_correct_0=int(cm[0, 0]),
            precision_1=prec[1], recall_1=rec[1], f1_1=f1[1], n_correct_1=int(cm[1, 1]),
            n_test=len(y_test), fpr=fpr, roc_auc=roc_auc, pr_auc=pr_auc,
            train_time_s=train_time_s, latency_ms=latency_ms,
        )
        print(f"{dname:10s}/{mname:12s} test_acc={test_acc:.3f} ROC-AUC={roc_auc:.3f} "
              f"PR-AUC={pr_auc:.3f} FPR={fpr:.3f} train_time={train_time_s:.2f}s "
              f"latency={latency_ms:.4f} ms/sample")


table5 = save_table(pd.DataFrame([
    {"Model": m, "Dataset": d, "Training Score": round(results[d][m]["train_accuracy"], 4)}
    for d in DATASETS for m in MODEL_FACTORY
]), "Table5_training_scores")

table6 = save_table(pd.DataFrame([
    {"Accuracy Result": round(results[d]["XGBoost"]["test_accuracy"], 4), "Data Sets": f"{d} data"}
    for d in DATASETS
]), "Table6_XGBoost_accuracy")

def per_class_table(dname):
    r = results[dname]["XGBoost"]
    rows = [
        {"Data Sets": 0, "Precision": round(r["precision_0"], 2), "Recall": round(r["recall_0"], 2),
         "F1-score": round(r["f1_0"], 2), "Number of Correct Answers": r["n_correct_0"]},
        {"Data Sets": 1, "Precision": round(r["precision_1"], 2), "Recall": round(r["recall_1"], 2),
         "F1-score": round(r["f1_1"], 2), "Number of Correct Answers": r["n_correct_1"]},
        {"Data Sets": "Accuracy", "Precision": "-", "Recall": "-",
         "F1-score": round(r["test_accuracy"], 2),
         "Number of Correct Answers": r["n_correct_0"] + r["n_correct_1"]},
        {"Data Sets": "ROC-AUC", "Precision": "-", "Recall": "-",
         "F1-score": round(r["roc_auc"], 3), "Number of Correct Answers": "-"},
        {"Data Sets": "PR-AUC", "Precision": "-", "Recall": "-",
         "F1-score": round(r["pr_auc"], 3), "Number of Correct Answers": "-"},
        {"Data Sets": "False Positive Rate", "Precision": "-", "Recall": "-",
         "F1-score": round(r["fpr"], 3), "Number of Correct Answers": "-"},
    ]
    return pd.DataFrame(rows)

table7 = save_table(per_class_table("Universal"), "Table7_universal_full_metrics_XGBoost")
table8 = save_table(per_class_table("Dependent"), "Table8_dependent_full_metrics_XGBoost")

table9 = save_table(pd.DataFrame([
    {"RF Accuracy": round(results[d]["RandomForest"]["test_accuracy"], 4), "Data Sets": f"{d} Data"}
    for d in DATASETS
]), "Table9_RandomForest_accuracy")

table10 = save_table(pd.DataFrame([
    {"Hist Gradient Boosting Accuracy": round(results[d]["HistGB"]["test_accuracy"], 4), "Data Sets": f"{d} Data"}
    for d in DATASETS
]), "Table10_HistGB_accuracy")

table11 = save_table(pd.DataFrame([
    {"Data Sets": f"Accuracy in {d} Data",
     "XGBoost": round(results[d]["XGBoost"]["test_accuracy"], 4),
     "Random Forest": round(results[d]["RandomForest"]["test_accuracy"], 4),
     "Hist Gradient Boosting": round(results[d]["HistGB"]["test_accuracy"], 4)}
    for d in DATASETS
]), "Table11_comparative_accuracy")

table14 = save_table(pd.DataFrame([
    {"Dataset": d, "Model": m,
     "Training Time (s)": round(results[d][m]["train_time_s"], 3),
     "Inference Latency (ms/sample)": round(results[d][m]["latency_ms"], 4),
     "Test Accuracy": round(results[d][m]["test_accuracy"], 4)}
    for d in DATASETS for m in MODEL_FACTORY
]), "Table14_efficiency_comparison")

table15 = save_table(pd.DataFrame([
    {"Dataset": d, "Model": m,
     "Accuracy": round(results[d][m]["test_accuracy"], 4),
     "ROC-AUC": round(results[d][m]["roc_auc"], 4),
     "PR-AUC": round(results[d][m]["pr_auc"], 4),
     "FPR": round(results[d][m]["fpr"], 4),
     "Precision (class 1)": round(results[d][m]["precision_1"], 4),
     "Recall (class 1)": round(results[d][m]["recall_1"], 4),
     "F1 (class 1)": round(results[d][m]["f1_1"], 4)}
    for d in DATASETS for m in MODEL_FACTORY
]), "Table15_expanded_metrics_all_models")


print("\n" + "=" * 80)
print("STEP 6 — FEATURE IMPORTANCE")
print("=" * 80)

fi_tables = {}
for dname, (df, feats) in DATASETS.items():
    X_train, X_test, y_train, y_test = splits[dname]
    pipe = build_pipeline(MODEL_FACTORY["XGBoost"]())
    pipe.fit(X_train, y_train)
    importances = pipe.named_steps["model"].feature_importances_
    fi = pd.DataFrame({"Feature": feats, "Importance": importances}).sort_values(
        "Importance", ascending=False
    ).reset_index(drop=True)
    fi["Rank"] = fi.index + 1
    fi["Importance"] = fi["Importance"].round(4)
    fi_tables[dname] = fi
    save_table(fi, f"Table2_feature_importance_{dname}")
    print(f"\nTop 5 — {dname} dataset:")
    print(fi.head(5).to_string(index=False))


print("\n" + "=" * 80)
print("STEP 7 — GENERATING FIGURES")
print("=" * 80)



fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, dname, n_top in zip(axes, ("Universal", "Dependent"), (10, 5)):
    fi = fi_tables[dname].head(n_top).iloc[::-1]
    bars = ax.barh(fi["Feature"], fi["Importance"], color=COLORS["XGBoost"], alpha=0.85)
    ax.set_xlabel("XGBoost Gain Importance")
    ax.set_title(f"{dname} Dataset (top {n_top} of {len(fi_tables[dname])} features)")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, fi["Importance"].max() * 1.2)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Figure 1. Feature Importance by Dataset (computed independently — "
             "Universal traffic features and Dependent semantic features are never mixed)",
             fontsize=11, y=1.03)
fig.tight_layout()
save_fig(fig, "Figure1_feature_importance")


metrics_for_fig3 = ["Precision (class 1)", "Recall (class 1)", "F1 (class 1)", "ROC-AUC"]
fig, ax = plt.subplots(figsize=(7.5, 5))
x = np.arange(len(DATASETS))
width = 0.2
xgb_rows = table15[table15["Model"] == "XGBoost"].set_index("Dataset")
for i, metric in enumerate(metrics_for_fig3):
    vals = [xgb_rows.loc[d, metric] for d in DATASETS]
    bars = ax.bar(x + (i - 1.5) * width, vals, width, label=metric)
    ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(list(DATASETS.keys()))
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Figure 3. XGBoost Performance Metrics — Universal vs Dependent Dataset")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_fig(fig, "Figure3_performance_metrics")


fig, ax = plt.subplots(figsize=(7.5, 5))
x = np.arange(len(DATASETS))
width = 0.25
for i, mname in enumerate(MODEL_FACTORY):
    vals = [results[d][mname]["test_accuracy"] for d in DATASETS]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=mname, color=COLORS[mname])
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(list(DATASETS.keys()))
ax.set_ylim(0, 1.1)
ax.set_ylabel("Test Accuracy")
ax.set_title("Figure 4. Comparison of Model Accuracy — Universal vs Dependent Dataset")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_fig(fig, "Figure4_model_accuracy_comparison")



LITERATURE = {
    "PTABLE": 0.5643, "SOM": 0.7689, "SMART": 0.9260,
    "SSOM": 0.7314, "CBF": 0.9601, "SS": 0.9523,
}
own_universal = {m: results["Universal"][m]["test_accuracy"] for m in MODEL_FACTORY}
methods = list(LITERATURE.keys()) + [f"{m} (this study)" for m in MODEL_FACTORY]
values = list(LITERATURE.values()) + [own_universal[m] for m in MODEL_FACTORY]
bar_colors = ["#B0B0B0"] * len(LITERATURE) + [COLORS[m] for m in MODEL_FACTORY]

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.bar(methods, values, color=bar_colors)
ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2, rotation=0)
ax.set_ylabel("Accuracy (Universal Data)")
ax.set_ylim(0, 1.15)
ax.set_title("Figure 5. Comparison of Classification Methods for Automated Robot Detection\n"
             "(gray = prior methods from Lagopoulos & Tsoumakas, 2020; colored = this study)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_fig(fig, "Figure5_comparison_literature_methods")


fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
for ax, dname in zip(axes, DATASETS):
    data = [cv_scores[dname][m] for m in MODEL_FACTORY]
    bp = ax.boxplot(data, tick_labels=list(MODEL_FACTORY.keys()), patch_artist=True, widths=0.5)
    for patch, mname in zip(bp["boxes"], MODEL_FACTORY):
        patch.set_facecolor(COLORS[mname])
        patch.set_alpha(0.6)
    ax.set_title(f"{dname} Dataset")
    ax.set_ylabel("CV Accuracy (5 folds)")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Figure 6. Distribution of 5-Fold Cross-Validation Accuracy by Model "
             "(see Table 13 for paired significance tests)", fontsize=11, y=1.03)
fig.tight_layout()
save_fig(fig, "Figure6_cv_significance_boxplot")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
x = np.arange(len(DATASETS))
width = 0.25
for i, mname in enumerate(MODEL_FACTORY):
    tvals = [results[d][mname]["train_time_s"] for d in DATASETS]
    bars = axes[0].bar(x + (i - 1) * width, tvals, width, label=mname, color=COLORS[mname])
    axes[0].bar_label(bars, fmt="%.2f", fontsize=7, padding=2)
axes[0].set_xticks(x); axes[0].set_xticklabels(list(DATASETS.keys()))
axes[0].set_ylabel("Training Time (s)")
axes[0].set_title("Training Time")
axes[0].spines[["top", "right"]].set_visible(False)

for i, mname in enumerate(MODEL_FACTORY):
    lvals = [results[d][mname]["latency_ms"] for d in DATASETS]
    bars = axes[1].bar(x + (i - 1) * width, lvals, width, label=mname, color=COLORS[mname])
    axes[1].bar_label(bars, fmt="%.4f", fontsize=7, padding=2)
axes[1].set_xticks(x); axes[1].set_xticklabels(list(DATASETS.keys()))
axes[1].set_ylabel("Inference Latency (ms/sample)")
axes[1].set_title("Inference Latency")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].legend(loc="upper center", bbox_to_anchor=(-0.15, -0.12), ncol=3, frameon=False, fontsize=9)

fig.suptitle("Figure 7. Computational Efficiency Comparison (supports the efficiency-focused "
             "narrative requested in review) ", fontsize=11, y=1.03)
fig.tight_layout()
save_fig(fig, "Figure7_efficiency_comparison")


print("\n" + "=" * 80)
print("STEP 8 — MANUSCRIPT-READY NUMBERS")
print("=" * 80)

for d in DATASETS:
    for m in MODEL_FACTORY:
        r = results[d][m]
        print(f"{d} / {m}: accuracy={r['test_accuracy']:.3f}, "
              f"ROC-AUC={r['roc_auc']:.3f}, PR-AUC={r['pr_auc']:.3f}, "
              f"FPR={r['fpr']:.3f}, train_time={r['train_time_s']:.2f}s, "
              f"latency={r['latency_ms']:.4f} ms/sample")

print("\nStatistical significance (paired t-test, XGBoost vs others):")
print(table13[["Dataset", "Comparison", "t-test p-value", "Significant at alpha=0.05"]]
      .to_string(index=False))


print("\n" + "=" * 80)
print("STEP 9 — PACKAGING OUTPUTS")
print("=" * 80)

zip_path = os.path.join(OUT_DIR, "revision_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_ in os.walk(OUT_DIR):
        for fname in files_:
            if fname == "revision_outputs.zip":
                continue
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, OUT_DIR)
            zf.write(fpath, arcname)
print(f"All tables and figures zipped to: {zip_path}")

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except Exception as e:
    print("Could not auto-trigger download (not running in Colab?). "
          f"Find the file manually at {zip_path}. ({e})")

print("\nDONE.")


STEP 1 — LOADING DATA
Universal dataset : 67,352 sessions, 30 features 30, see revision guide)
Dependent dataset : 67,352 sessions, 5 features
Class balance (both datasets share the same ROBOT labels): {0: 53858, 1: 13494} (0 = human, 1 = robot)

STEP 2 — TRAIN/TEST SPLIT
Universal : total=67,352  train=53,881  test=13,471  (exact 20% = 13,470)
Dependent : total=67,352  train=53,881  test=13,471  (exact 20% = 13,470)
  -> saved /content/revision_outputs/tables/TableS1_split_reconciliation.csv
  -> saved /content/revision_outputs/tables/Table3_hyperparameters.csv

STEP 3 — 5-FOLD CROSS-VALIDATION
Universal  | XGBoost      | folds = [0.9765, 0.9775, 0.9761, 0.977, 0.9787] | mean = 0.9771 | std = 0.0009
Universal  | RandomForest | folds = [0.968, 0.9661, 0.9662, 0.9675, 0.9675] | mean = 0.9671 | std = 0.0008
Universal  | HistGB       | folds = [0.9751, 0.9748, 0.9745, 0.9746, 0.9751] | mean = 0.9748 | std = 0.0003
Dependent  | XGBoost      | folds = [0.8398, 0.8421, 0.8422, 0.8363, 0.846

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DONE.
